<a href="https://colab.research.google.com/github/LeonardoFisAct/DSML_Notes/blob/main/Deep_agent_ins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --q -U langchain langchain-openrouter deepagents langchain-community langchain-core langgraph actuarialmath

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.6/72.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 704.7/704.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 44.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import os
#from getpass import getpass
from google.colab import userdata

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENR_API_KEY')

In [3]:
import actuarialmath as am
from typing import Literal
from langchain_core.tools import tool

In [4]:
# ==========================================
# 1. ACTUARIAL LOGIC (THE TOOL)
# ==========================================

# We define the helper class to hold the state of the tables
class ActuarialEngine:
    def __init__(self):
        # Load Standard Ultimate Life Table (SULT)
        self.life = am.SULT()

    def calculate_insurance_value(self,
                                  insurance_type: Literal['whole_life', 'term', 'endowment', 'pure_endowment','deferred_insurance'],
                                  age: int,
                                  duration: int = 0,
                                  benefit: float = 1.0) -> dict:
        """
        Calculates the Actuarial Present Value (APV) for various insurance products based on the SULT.

        Args:
            insurance_type: Type of insurance. Options: 'whole_life', 'term', 'endowment','pure_endowment','deferred_insurance'.
            age: The age of the insured person (x).
            duration: The term of the policy in years (n/t). Required for term,endowment,pure_endowment and deferred_insurance . Use 0 for whole_life.
            benefit: The benefit amount (b). Default is 1.

        Returns:
            dict: Contains the calculated value and a description of the symbol.
        """
        try:
            val = 0.0
            symbol = ""

            if insurance_type == 'whole_life':
                val = self.life.whole_life_insurance(x=age, b=benefit)
                symbol = f"A_{{{age}}}"
            elif insurance_type == 'term':
                val = self.life.term_insurance(x=age, t=duration, b=benefit)
                symbol = f"A^1_{{{age}:{duration}}}"
            elif insurance_type == 'endowment':
                val = self.life.endowment_insurance(x=age, t=duration, b=benefit)
                symbol = f"A_{{{age}:{duration}}}"
            elif insurance_type == 'pure_endowment':
                val = self.life.E_x(x=age, t=duration)*benefit
                symbol = f"_{{{duration}}}E_{{{age}}}"
            elif insurance_type == 'deferred_insurance':
                val = self.life.deferred_insurance(x=age, u=duration, b=benefit)
                symbol = f"_{{{duration}}}|A_{{{age}}}"
            else:
                return {"error": f"Unknown insurance type: {insurance_type}"}

            return {
                "value": float(val),
                "symbol": symbol,
                "parameters": {"type": insurance_type, "age": age, "duration": duration, "benefit": benefit}
            }
        except Exception as e:
            return {"error": str(e)}

# Instantiate the engine
engine = ActuarialEngine()

In [6]:
print(f"💳 Test: {engine.calculate_insurance_value('whole_life',40,benefit=100000)}")
print(f"💳 Test: {engine.calculate_insurance_value('term',40,20,100000)}")
print(f"💳 Test: {engine.calculate_insurance_value('endowment',40,20,100000)}")
print(f"💳 Test: {engine.calculate_insurance_value('pure_endowment',40,20,100000)}")
print(f"💳 Test: {engine.calculate_insurance_value('deferred_insurance',40,20,100000)}")

💳 Test: {'value': 12105.921086937971, 'symbol': 'A_{40}', 'parameters': {'type': 'whole_life', 'age': 40, 'duration': 0, 'benefit': 100000}}
💳 Test: {'value': 1463.3042757845833, 'symbol': 'A^1_{40:20}', 'parameters': {'type': 'term', 'age': 40, 'duration': 20, 'benefit': 100000}}
💳 Test: {'value': 38126.309052437246, 'symbol': 'A_{40:20}', 'parameters': {'type': 'endowment', 'age': 40, 'duration': 20, 'benefit': 100000}}
💳 Test: {'value': 36663.00477665266, 'symbol': '_{20}E_{40}', 'parameters': {'type': 'pure_endowment', 'age': 40, 'duration': 20, 'benefit': 100000}}
💳 Test: {'value': 10642.616811153388, 'symbol': '_{20}|A_{40}', 'parameters': {'type': 'deferred_insurance', 'age': 40, 'duration': 20, 'benefit': 100000}}


In [7]:
from langchain_openrouter import ChatOpenRouter

#Avialable free models (May 28,2026)
deepseek_v4 = "deepseek/deepseek-v4-flash:free"
kimik2_6 = "moonshotai/kimi-k2.6:free"
gemm4_3b = "google/gemma-4-31b-it:free"
nemotron3_super = "nvidia/nemotron-3-super-120b-a12b:free"
nemotron3_nano =  "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
nemotron3_ultra = 'nvidia/nemotron-3-ultra-550b-a55b:free'

model = ChatOpenRouter(
    model=nemotron3_ultra
)


In [8]:
from deepagents import create_deep_agent

SYSTEM_PROMPT = """
You are an expert Actuary. You have access to a tool called `calculate_insurance_value`
that uses the Standard Ultimate Life Table (SULT) with i=0.05.

    1. Always use the tool for calculations. Do not guess values.
    2. When the tool returns a value, present it nicely (e.g., "The APV is...").
    3. If the user asks for actuarial notation (like Ax), use LaTeX formatting in your response.
    4. If the tool returns a 'symbol', display it in LaTeX.

If the tool returns status "error", explain the issue to the user clearly.
"""

agent = create_deep_agent(
    model=model,
    tools=[engine.calculate_insurance_value],
    system_prompt=SYSTEM_PROMPT
    ).with_config({"recursion_limit": 20})  #recursion_limit limits the number of steps the agent will run



In [9]:
type(agent)

langgraph.graph.state.CompiledStateGraph

In [10]:

user_question = "I need a term insurance with a duration of 20 for a 40-year-old and a benefit of $100,000."

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_question,
            }
        ],
    }
)

# Extract and print the agent's response
if "messages" in response and len(response["messages"]) > 0:
    ai_message = response["messages"][-1]
    print(ai_message.content)
else:
    print("No response received from the agent.")


The APV for a 20-year term insurance on a 40-year-old with a $100,000 benefit is:

**$1,463.30**

Actuarial notation: $A^{1}_{40:\overline{20}|}$


In [11]:
response

{'messages': [HumanMessage(content='I need a term insurance with a duration of 20 for a 40-year-old and a benefit of $100,000.', additional_kwargs={}, response_metadata={}, id='689a4f51-ae11-4bba-889d-cd26ed69abfd'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants a term insurance calculation with:\n- Age: 40\n- Duration: 20 years\n- Benefit: $100,000\n\nI need to use the calculate_insurance_value tool with insurance_type="term", age=40, duration=20, benefit=100000.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a term insurance calculation with:\n- Age: 40\n- Duration: 20 years\n- Benefit: $100,000\n\nI need to use the calculate_insurance_value tool with insurance_type="term", age=40, duration=20, benefit=100000.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b-20260604:free', 'id': 'gen-1782414240-o3EFaUsaUmcuc0KDK9jF', 'created': 1782414240, 'object': 'chat.completion

In [12]:

user_question = "Which insurance is cheaper for a 40 years old person with a benefit of $100,000; a whole life insurance or a 20 years deferred insurance? "

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_question,
            }
        ],
    }
)

# Extract and print the agent's response
if "messages" in response and len(response["messages"]) > 0:
    ai_message = response["messages"][-1]
    print(ai_message.content)
else:
    print("No response received from the agent.")

The **20-year deferred insurance** is cheaper.

| Insurance Type | APV (Actuarial Present Value) |
|----------------|-------------------------------|
| Whole Life ($A_{40}$) | **$12,105.92** |
| 20-Year Deferred ($_{20}\|A_{40}$) | **$10,642.62** |

The deferred insurance costs about **$1,463 less** because the benefit is only paid if the insured survives 20 years *and* then dies—so the insurer's expected payout is lower.


In [14]:
12_105.92 - 10_642.62

1463.2999999999993

In [13]:
response

{'messages': [HumanMessage(content='Which insurance is cheaper for a 40 years old person with a benefit of $100,000; a whole life insurance or a 20 years deferred insurance? ', additional_kwargs={}, response_metadata={}, id='7ce6765b-679d-4a71-baf1-614d102d5c83'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking me to compare the cost of two insurance products for a 40-year-old person with a $100,000 benefit:\n1. Whole life insurance\n2. 20-year deferred insurance\n\nI need to use the calculate_insurance_value tool for both types. Let me call the tool for both.\n\nFor whole life insurance: insurance_type="whole_life", age=40, benefit=100000, duration=0 (or default)\nFor deferred insurance: insurance_type="deferred_insurance", age=40, benefit=100000, duration=20', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking me to compare the cost of two insurance products for a 40-year-old person with a 